# Evaluating Lightweight Granite 4.0 H 350M for Simulated Tool Use

This notebook refactors the original JSON classification workflow into a simulated tool-use evaluation harness for:

`unsloth/granite-4.0-h-350m-GGUF`

The goal is not to call real external tools. The goal is to test whether a very small model can select the correct tool for a user request and return a compact, valid JSON object.

## Why simulated tool use?

Small local models can be useful as routing models when the problem is narrow:

1. Choose from a small tool list.
2. Return one flat JSON object.
3. Avoid long explanations.
4. Validate outputs with Python.
5. Measure selection accuracy before using the model in an agent pipeline.

## Model-size constraint

A 350M parameter model should not be asked to perform broad multi-step planning. This notebook keeps the evaluation intentionally small:

- Five tool labels plus `no_tool`.
- One required output field: `tool`.
- One optional short string field: `argument`.
- Deterministic decoding.
- Short prompts.
- Few-shot examples.
- Schema validation and retry.

The primary metric is correct tool selection.


## 1. Runtime setup

In Colab, choose:

`Runtime` -> `Change runtime type` -> `CPU`

This notebook uses CPU-friendly inference settings. A GPU is not required for the basic evaluation.


In [ ]:
# Install minimal dependencies.
# The extra index asks pip to prefer a prebuilt CPU wheel for llama-cpp-python when available.

!pip -q install --upgrade huggingface_hub jsonschema pandas tqdm matplotlib scikit-learn
!pip -q install --upgrade llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu


## 2. Download the GGUF model

The default model is the same lightweight Granite model family used in the original notebook.

The file listing step is included so the notebook remains usable if the repo changes its exact GGUF filenames.


In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files
from pathlib import Path

MODEL_REPO = "unsloth/granite-4.0-h-350m-GGUF"
PREFERRED_MODEL_FILE = "granite-4.0-h-350m-UD-Q4_K_XL.gguf"

print("Available GGUF files in the repo:")
gguf_files = [f for f in list_repo_files(MODEL_REPO) if f.endswith(".gguf")]
for f in gguf_files:
    print("-", f)

MODEL_FILE = PREFERRED_MODEL_FILE if PREFERRED_MODEL_FILE in gguf_files else gguf_files[0]

model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
)

print("\nUsing model file:", MODEL_FILE)
print("Downloaded model path:", model_path)
print("Size on disk MB:", round(Path(model_path).stat().st_size / 1_000_000, 1))


## 3. Load the model with conservative settings

Tool selection is a classification task. Use deterministic generation and a small context window.

For this model size, prefer:

- `temperature=0.0`
- `max_tokens` under 80
- `n_ctx=1024`
- short few-shot prompts
- flat JSON schemas


In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_ctx=1024,
    n_threads=2,
    n_gpu_layers=0,
    verbose=False,
)

print("Model loaded.")


## 4. Define the simulated tool catalog

These are not real tools. They are labels used to evaluate routing.

Keep the tool list short and separable. Tool names should be concrete and mutually exclusive.


In [ ]:
SIMULATED_TOOLS = {
    "calculator": "Use for arithmetic, counting, percentages, numeric comparisons, and unit-like math.",
    "weather_lookup": "Use for current weather or forecast requests for a place.",
    "email_search": "Use for finding, reading, summarizing, or checking email messages.",
    "calendar_lookup": "Use for checking events, availability, meetings, schedules, or calendar conflicts.",
    "file_search": "Use for finding, reading, or summarizing uploaded files, documents, notebooks, PDFs, or reports.",
    "no_tool": "Use when the request can be answered directly without any external tool.",
}

TOOL_NAMES = list(SIMULATED_TOOLS.keys())

for name, description in SIMULATED_TOOLS.items():
    print(f"{name}: {description}")


## 5. JSON schema for tool selection

The model only has to produce:

```json
{"tool":"calculator","argument":"20% of 85"}
```

The `argument` field is a short copy of the key input needed by the tool. The evaluation primarily grades the `tool` field.

This keeps the task realistic but still appropriate for a 350M model.


In [ ]:
TOOL_SELECTION_SCHEMA = {
    "type": "object",
    "properties": {
        "tool": {
            "type": "string",
            "enum": TOOL_NAMES,
        },
        "argument": {
            "type": "string",
            "maxLength": 160,
        },
    },
    "required": ["tool", "argument"],
    "additionalProperties": False,
}


## 6. Core utilities: generation, JSON extraction, validation, and repair

Treat model output as untrusted text.

The original notebook already used the right production pattern: generate, extract JSON, validate schema, safely coerce simple outputs, and retry. This refactor reuses that idea for tool selection.


In [ ]:
import json
import re
import time
from typing import Optional
from jsonschema import validate

def generate_text(prompt: str, max_tokens: int = 80, debug: bool = False) -> str:
    """Run a deterministic completion and return raw model text."""
    response = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=0.0,
        top_p=1.0,
        repeat_penalty=1.05,
        stop=["<|endoftext|>", "<|end_of_text|>", "\n\n\n"],
    )
    raw = response["choices"][0].get("text", "")
    if debug:
        print("RAW MODEL OUTPUT:")
        print(repr(raw))
    return raw.strip()


def extract_first_json_object(text: str) -> str:
    """Extract the first complete JSON object using brace counting."""
    if not text or not text.strip():
        raise ValueError("Empty model output")

    start = text.find("{")
    if start == -1:
        raise ValueError(f"No JSON object found in output: {text!r}")

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):
        ch = text[i]

        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return text[start:i + 1]

    raise ValueError(f"Started JSON object but did not find closing brace: {text!r}")


def coerce_tool_name(raw_text: str) -> Optional[dict]:
    """Convert a bare tool name into a valid object when exactly one tool is mentioned."""
    cleaned = raw_text.strip().lower()
    cleaned = cleaned.replace("```json", "").replace("```", "").strip()
    cleaned = cleaned.strip(" .,:;!?'\"`\\n\\t")

    if cleaned in TOOL_NAMES:
        return {"tool": cleaned, "argument": ""}

    hits = []
    for name in TOOL_NAMES:
        if re.search(rf"\\b{re.escape(name)}\\b", cleaned):
            hits.append(name)

    if len(hits) == 1:
        return {"tool": hits[0], "argument": ""}

    return None


def parse_and_validate_tool_output(raw_text: str) -> dict:
    """Parse model output into JSON and validate it against the tool selection schema."""
    try:
        json_text = extract_first_json_object(raw_text)
        data = json.loads(json_text)
        validate(instance=data, schema=TOOL_SELECTION_SCHEMA)
        return data
    except Exception as json_error:
        coerced = coerce_tool_name(raw_text)
        if coerced is not None:
            validate(instance=coerced, schema=TOOL_SELECTION_SCHEMA)
            return coerced
        raise json_error


def compact_schema_hint() -> str:
    return '{"tool":"calculator|weather_lookup|email_search|calendar_lookup|file_search|no_tool","argument":"short input for the chosen tool"}'


## 7. Prompt builder for simulated tool selection

The prompt is deliberately short and repetitive. The model should classify, not explain.

The most important cue is the final `JSON:` line.


In [ ]:
def build_tool_selection_prompt(user_request: str) -> str:
    tool_lines = "\n".join(
        f"- {name}: {description}"
        for name, description in SIMULATED_TOOLS.items()
    )

    return f"""
You are a strict tool selection function.
Return only valid JSON. No markdown. No explanation.

Choose exactly one tool.

Tools:
{tool_lines}

Rules:
- Use no_tool only when no external information or action is needed.
- If the user asks about an uploaded file or document, choose file_search.
- If the user asks about email, choose email_search.
- If the user asks about meetings, schedule, calendar, or availability, choose calendar_lookup.
- If the user asks about weather or forecast, choose weather_lookup.
- If the user asks for math, choose calculator.

Schema:
{compact_schema_hint()}

Examples:
User: "What is 18% of 240?"
JSON: {{"tool":"calculator","argument":"18% of 240"}}

User: "Will it rain in Tampa tomorrow?"
JSON: {{"tool":"weather_lookup","argument":"Tampa tomorrow"}}

User: "Find emails from Alex about the invoice."
JSON: {{"tool":"email_search","argument":"from Alex about the invoice"}}

User: "Do I have any meetings Friday afternoon?"
JSON: {{"tool":"calendar_lookup","argument":"Friday afternoon"}}

User: "Summarize the notebook I uploaded."
JSON: {{"tool":"file_search","argument":"uploaded notebook"}}

User: "Explain what JSON is in one sentence."
JSON: {{"tool":"no_tool","argument":"Explain what JSON is in one sentence."}}

User: {user_request!r}
JSON:
""".strip()


## 8. Tool selection with repair

The repair prompt asks the model to fix a malformed output into the required JSON shape. This is useful for small models that sometimes return a bare tool name or extra text.


In [ ]:
def select_tool(user_request: str, retries: int = 2, debug: bool = False) -> dict:
    """Select a simulated tool for a user request."""
    prompt = build_tool_selection_prompt(user_request)
    raw_outputs = []
    started = time.perf_counter()

    for attempt in range(retries + 1):
        raw = generate_text(prompt, max_tokens=70, debug=debug)
        raw_outputs.append(raw)

        try:
            parsed = parse_and_validate_tool_output(raw)
            parsed["latency_s"] = round(time.perf_counter() - started, 4)
            parsed["raw_output"] = raw
            parsed["attempts"] = attempt + 1
            return parsed
        except Exception as error:
            if debug:
                print(f"Attempt {attempt + 1} failed: {type(error).__name__}: {error}")

            prompt = f"""
Repair this tool selection output.
Return one valid JSON object only.
No markdown. No explanation.

User request:
{user_request!r}

Previous invalid output:
{raw!r}

Required schema:
{compact_schema_hint()}

Allowed tool names:
{", ".join(TOOL_NAMES)}

JSON:
""".strip()

    raise ValueError(f"Could not produce valid tool JSON. Raw outputs: {raw_outputs!r}")


print(select_tool("What is 15% of 80?", debug=True))


## 9. Create a compact labeled evaluation set

This dataset is intentionally small and concrete.

For a 350M model, start with 20 to 40 examples before expanding. Add difficult examples only after the basic routing task works.


In [ ]:
EVAL_SET = [
    {"request": "What is 42 plus 19?", "expected_tool": "calculator"},
    {"request": "Calculate 18% of 240.", "expected_tool": "calculator"},
    {"request": "Which is larger, 9.8 or 9.11?", "expected_tool": "calculator"},
    {"request": "Will it rain in Brandon tomorrow?", "expected_tool": "weather_lookup"},
    {"request": "Give me the weekend forecast for New York.", "expected_tool": "weather_lookup"},
    {"request": "What is the temperature in San Francisco right now?", "expected_tool": "weather_lookup"},
    {"request": "Find unread emails from Sarah about the contract.", "expected_tool": "email_search"},
    {"request": "Summarize my latest invoice emails.", "expected_tool": "email_search"},
    {"request": "Check whether I received an email from GitHub today.", "expected_tool": "email_search"},
    {"request": "Do I have any meetings tomorrow morning?", "expected_tool": "calendar_lookup"},
    {"request": "Find a free 30 minute slot next Wednesday.", "expected_tool": "calendar_lookup"},
    {"request": "What is on my calendar at 2pm?", "expected_tool": "calendar_lookup"},
    {"request": "Analyze the PDF I uploaded.", "expected_tool": "file_search"},
    {"request": "Summarize the attached notebook.", "expected_tool": "file_search"},
    {"request": "Find the section about latency in my report.", "expected_tool": "file_search"},
    {"request": "Explain the difference between a list and a tuple.", "expected_tool": "no_tool"},
    {"request": "Write a friendly thank you message.", "expected_tool": "no_tool"},
    {"request": "What does CPU inference mean?", "expected_tool": "no_tool"},
    {"request": "Check my email and tell me if I have meetings tomorrow.", "expected_tool": "email_search"},
    {"request": "Look at the uploaded benchmark log and calculate the average latency.", "expected_tool": "file_search"},
]

print("Evaluation examples:", len(EVAL_SET))


## 10. Batch evaluation

Metrics:

- `valid_json_rate`: percent of examples that returned schema-valid JSON.
- `tool_accuracy`: percent of examples where `tool` matched the expected label.
- `avg_latency_s`: average end-to-end inference time per request.
- `repair_rate`: percent of examples that needed more than one generation attempt.

For routing models, invalid JSON is a failure even if the raw text contains the right idea.


In [ ]:
import pandas as pd
from tqdm.auto import tqdm

def evaluate_tool_selection(eval_set, debug: bool = False) -> pd.DataFrame:
    rows = []

    for item in tqdm(eval_set):
        request = item["request"]
        expected = item["expected_tool"]

        try:
            prediction = select_tool(request, retries=2, debug=debug)
            predicted_tool = prediction["tool"]
            rows.append({
                "request": request,
                "expected_tool": expected,
                "predicted_tool": predicted_tool,
                "correct": predicted_tool == expected,
                "valid_json": True,
                "argument": prediction.get("argument", ""),
                "latency_s": prediction.get("latency_s"),
                "attempts": prediction.get("attempts"),
                "raw_output": prediction.get("raw_output"),
                "error": "",
            })
        except Exception as error:
            rows.append({
                "request": request,
                "expected_tool": expected,
                "predicted_tool": None,
                "correct": False,
                "valid_json": False,
                "argument": "",
                "latency_s": None,
                "attempts": None,
                "raw_output": "",
                "error": f"{type(error).__name__}: {error}",
            })

    return pd.DataFrame(rows)

results_df = evaluate_tool_selection(EVAL_SET)
display(results_df)

valid_json_rate = results_df["valid_json"].mean()
tool_accuracy = results_df["correct"].mean()
avg_latency = results_df["latency_s"].dropna().mean()
repair_rate = (results_df["attempts"].fillna(0) > 1).mean()

print("Valid JSON rate:", round(valid_json_rate, 3))
print("Tool accuracy:", round(tool_accuracy, 3))
print("Average latency seconds:", round(avg_latency, 3) if pd.notna(avg_latency) else None)
print("Repair rate:", round(repair_rate, 3))


## 11. Confusion matrix

A confusion matrix shows where the model confuses tools.

For example, `calendar_lookup` versus `email_search` can be difficult when a request mentions both email and meetings. Decide your priority rule before evaluation.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

labels = TOOL_NAMES

plot_df = results_df.copy()
plot_df["predicted_tool_for_plot"] = plot_df["predicted_tool"].fillna("invalid")

plot_labels = labels + ["invalid"]

cm = confusion_matrix(
    plot_df["expected_tool"],
    plot_df["predicted_tool_for_plot"],
    labels=plot_labels,
)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=plot_labels)
fig, ax = plt.subplots(figsize=(9, 7))
disp.plot(ax=ax, xticks_rotation=45)
plt.title("Simulated Tool Selection Confusion Matrix")
plt.show()


## 12. Inspect failures

Run failed or incorrect rows with `debug=True`.

For small models, common failure modes are:

1. Returning a bare tool name.
2. Returning extra explanation text.
3. Choosing `no_tool` when a tool is required.
4. Confusing email search and calendar lookup.
5. Confusing file search and direct answering.
6. Producing malformed JSON.


In [ ]:
failures_df = results_df[(results_df["valid_json"] == False) | (results_df["correct"] == False)]
display(failures_df[["request", "expected_tool", "predicted_tool", "raw_output", "error"]])

if len(failures_df) > 0:
    example_request = failures_df.iloc[0]["request"]
    print("Debugging first failure:")
    print(example_request)
    print(select_tool(example_request, debug=True))
else:
    print("No failures found in this run.")


## 13. Optional: add hard negatives

Hard negatives help test whether the model overuses tools.

These are requests that sound agentic but should still be `no_tool` because the user is asking for an explanation or draft, not an external action.


In [ ]:
HARD_NEGATIVES = [
    {"request": "Explain how a weather API works.", "expected_tool": "no_tool"},
    {"request": "Write an example email asking for a refund.", "expected_tool": "no_tool"},
    {"request": "Describe how calendar scheduling algorithms work.", "expected_tool": "no_tool"},
    {"request": "What is a file index?", "expected_tool": "no_tool"},
    {"request": "Show me how to calculate a percentage in Python.", "expected_tool": "no_tool"},
]

hard_negative_df = evaluate_tool_selection(HARD_NEGATIVES)
display(hard_negative_df)

print("Hard negative accuracy:", round(hard_negative_df["correct"].mean(), 3))


## 14. Optional: simulate tool execution

After the model selects a tool, you can simulate execution with placeholder functions.

This keeps the notebook safe and reproducible. No real email, calendar, weather, or file APIs are called.


In [ ]:
def simulated_calculator(argument: str) -> str:
    return f"[simulated calculator] Would calculate: {argument}"

def simulated_weather_lookup(argument: str) -> str:
    return f"[simulated weather] Would look up weather for: {argument}"

def simulated_email_search(argument: str) -> str:
    return f"[simulated email] Would search email for: {argument}"

def simulated_calendar_lookup(argument: str) -> str:
    return f"[simulated calendar] Would check calendar for: {argument}"

def simulated_file_search(argument: str) -> str:
    return f"[simulated file search] Would search files for: {argument}"

def simulated_no_tool(argument: str) -> str:
    return f"[no tool] Direct response should be generated for: {argument}"

SIMULATED_TOOL_FUNCTIONS = {
    "calculator": simulated_calculator,
    "weather_lookup": simulated_weather_lookup,
    "email_search": simulated_email_search,
    "calendar_lookup": simulated_calendar_lookup,
    "file_search": simulated_file_search,
    "no_tool": simulated_no_tool,
}

def route_and_simulate(user_request: str, debug: bool = False) -> dict:
    selection = select_tool(user_request, debug=debug)
    tool_name = selection["tool"]
    argument = selection.get("argument", "")
    simulated_result = SIMULATED_TOOL_FUNCTIONS[tool_name](argument)
    return {
        "request": user_request,
        "selection": selection,
        "simulated_result": simulated_result,
    }

route_and_simulate("Find unread emails from Sarah about the contract.")


## 15. Design guidance for 350M tool-selection models

Use this checklist before expanding the benchmark:

- Start with 3 to 6 tools.
- Keep tool descriptions short.
- Use mutually exclusive tool names.
- Avoid multi-tool planning in the first benchmark.
- Avoid nested argument schemas in early tests.
- Require one flat JSON object.
- Put the most important routing rules near the end of the prompt.
- Keep examples short and balanced across labels.
- Include hard negatives for `no_tool`.
- Grade JSON validity separately from tool accuracy.
- Inspect raw outputs before changing the prompt.
- Add tools only after the base confusion matrix is stable.

A 350M model can be a useful first-stage router, but it should be wrapped in validation, retries, and deterministic policies.


## 16. Next experiments

Recommended follow-up experiments:

1. Reduce the tool set to three labels and compare accuracy.
2. Expand from 20 examples to 100 examples.
3. Add a two-stage router:
   - Stage 1: `tool_required` versus `no_tool`
   - Stage 2: choose the tool only if required
4. Compare argument extraction accuracy separately from tool selection.
5. Track latency and accuracy across quantization files.
6. Test prompt variants with and without few-shot examples.

The safest production pattern is a small model router plus strict schema validation and a deterministic fallback policy.
